In [54]:
import pandas as pd
import pandas_gbq
from sqlalchemy import create_engine
import urllib.parse
# import pymssql
from datetime import datetime
import numpy as np
from google.cloud import bigquery
import numpy as np

| Município           | Código na SEF-MG | Código TOM-SERPRO |
|---------------------|------------------|-------------------|
| ALTO ALEGRE         | 001              | 0305              |
| AMAJARI             | 009              | 0026              |
| BOA VISTA           | 002              | 0301              |
| BONFIM              | 003              | 0307              |
| CANTA               | 010              | 0028              |
| CARACARAI           | 004              | 0303              |
| CAROEBE             | 011              | 0030              |
| IRACEMA             | 012              | 0032              |
| MUCAJAI             | 005              | 0309              |
| NORMANDIA           | 006              | 0311              |
| PACARAIMA           | 013              | 0034              |
| RORAINOPOLIS        | 014              | 0036              |
| SAO JOAO DA BALIZA  | 007              | 0313              |
| SAO LUIZ            | 008              | 0315              |
| UIRAMUTA            | 015              | 0038              |


In [96]:
project_id = "infra-itaborai"

queryrfb = """

SELECT 
    -- a.SITUACAO_CADASTRAL,e.DESCRICAO_SITUACAO,
    -- B.QUALIFICACAO_RESPONSAVEL,f.descricao,
    -- b.COD_PORTE_EMPRESA,g.desc_porte,
    -- a.pais,h.nome_pais,
    -- b.COD_NATUREZA_JUR,
    -- i.desc_natureza_juridica,
    -- a.municipio,
    -- j.descricao,
    -- a.motivo_situacao,
    -- k.desc_motivo,

    --UPPER (CONCAT(a.CNPJ_COMPLETO," - ",b.RAZAO_SOCIAL," - ",IFNULL(a.NOME_FANTASIA, " "), " - ",l.descricao)) AS CNPJ_RAZAO_SOCIAL_CNAE_PRINCIPAL,
    --UPPER(CONCAT(a.CNPJ_COMPLETO, " - ", b.RAZAO_SOCIAL)) AS CNPJ_RAZAO,
    a.CNPJ_COMPLETO AS CNPJ_COMPLETO,
    UPPER(REGEXP_REPLACE(b.RAZAO_SOCIAL, r'^[0-9.\s]+', '')) AS razao_social,
    --UPPER(a.NOME_FANTASIA) AS NOME_FANTASIA,
    --a.CNPJ_BASICO AS CNPJ_BASICO,
    --a.CNPJ_ORDEM AS CNPJ_ORDEM,
    --a.CNPJ_DV AS CNPJ_DV ,
    --a.MATRIZ_FILIAL AS MATRIZ_FILIAL ,
    
    --a.SITUACAO_CADASTRAL AS SITUACAO_CADASTRAL ,
    --UPPER(e.DESCRICAO_SITUACAO) AS DESCRICAO_SITUACAO,
    --a.DATA_SITUACAO_CADASTRAL AS DATA_SITUACAO_CADASTRAL,
    --a.MOTIVO_SITUACAO AS MOTIVO_SITUACAO,
    --UPPER(k.desc_motivo) AS DESC_SITUACAO,
    --UPPER(a.NOME_CIDADE_EXT)AS NOME_CIDADE_EXT,
    --UPPER(a.PAIS) AS PAIS ,
    --UPPER(h.descricao) AS DESC_PAIS,
    --a.Data_Inicio_Atividade AS DATA_INICIO_ATIVIDADE,
    --a.CNAE_PRINCIPAL AS CNAE_PRINCIPAL,

    
    --a.CNAE_SECUNDARIA AS CNAE_SECUNDARIA,
    --CONCAT(a.TIPO_LOGRADOURO, " ", a.LOGRADOURO, " ", a.NUM, " ", a.COMPLEMENTO, " ", a.BAIRRO, " ", a.CEP, " ", a.UF, " ", a.MUNICIPIO) AS ENDERECO_COMPLETO,
    --UPPER(a.TIPO_LOGRADOURO) AS TIPO_LOGRADOURO ,
    --UPPER(a.LOGRADOURO) AS LOGRADOURO,
    --a.NUM AS NUM,
    --UPPER(a.COMPLEMENTO) AS COMPLEMENTO,
    --UPPER(a.BAIRRO) AS BAIRRO,
    --a.CEP AS CEP,
    --UPPER(a.UF) AS UF,
    --a.MUNICIPIO AS MUNICIPIO,
    --j.descricao AS DESC_MUNICIPIO,
    CONCAT(A.DDD1,A.TEL1) AS telefone,
    --CONCAT(A.DDD2,A.TEL2) AS TELEFONE2,
    --a.DDD1 AS DDD1,
    --a.TEL1 AS TEL1,
    --a.DDD2 AS DDD2,
    --a.TEL2 AS TEL2,
    --a.DDD_FAX AS DDD_FAX,
    --a.TEL_FAX AS TEL_FAX,
    LOWER(a.E_MAIL) AS email,
    --UPPER(a.SITUACAO_ESPECIAL) AS SITUACAO_ESPECIAL ,
    --a.DATA_SIT_ESPECIAL AS DATA_SIT_ESPECIAL,

    --b.COD_NATUREZA_JUR AS COD_NATUREZA_JUR,
    --UPPER(i.descricao) AS DESC_NATUREZA_JURIDICA,
    --b.QUALIFICACAO_RESPONSAVEL AS QUALIFICACAO_RESPONSAVEL,
    --UPPER(f.descricao) AS DESC_RESPONSAVEL,
    --CAST(REPLACE(b.CAPITAL_SOCIAL, ',', '.') AS FLOAT64) AS CAPITAL_SOCIAL,
    --b.COD_PORTE_EMPRESA AS COD_PORTE_EMPRESA,
    --UPPER(g.desc_porte) AS DESC_PORTE,
    --UPPER(b.ENTE_RESPONSAVEL) AS ENTE_RESPONSAVEL,



    -- c.CNPJ_BASICO,
    --UPPER(c.OPCAO_PELO_SIMPLES) AS OPCAO_PELO_SIMPLES,
--PARSE_DATE('%Y%m%d', NULLIF(c.DATA_OPCAO_SIMPLES, '00000000')) AS DATA_OPCAO_SIMPLES,
    --PARSE_DATE('%Y%m%d', NULLIF(c.DATA_EXCLUSAO_DO_SIMPLES, '00000000')) AS DATA_EXCLUSAO_DO_SIMPLES,
    --UPPER(c.OPCAO_PELO_MEI) AS OPCAO_PELO_MEI,
    --PARSE_DATE('%Y%m%d', NULLIF(c.DATA_OPCAO_MEI, '00000000'))         AS DATA_OPCAO_MEI,
    --PARSE_DATE('%Y%m%d', NULLIF(c.DATA_EXCLUSAO_DO_MEI, '00000000'))     AS DATA_EXCLUSAO_DO_MEI,


    d.cnae AS CNAE,
    UPPER(d.descricao) AS DESCRICAO_CNAE
    -- d_anexo AS ANEXO,
    --UPPER(d.tipo) AS CLASIFICACAO_PRINCIPAL
    
    
FROM `mds_cnpj.estabelecimentos` AS a
    LEFT JOIN `mds_cnpj.empresa` AS b on a.cnpj_BASICO = b.cnpj_basico
    LEFT JOIN `mds_cnpj.simples_nacional` AS c on a.cnpj_BASICO = c.cnpj_basico
    LEFT JOIN `mds_cnpj.cnae_servico` AS d on a.CNAE_PRINCIPAL = d.cnae
    LEFT JOIN infra-itaborai.mds_cnpj.d_situacao_cadastral AS e on a.SITUACAO_CADASTRAL = e.id_situacao
    LEFT JOIN infra-itaborai.mds_cnpj.d_qualifsocio as f on B.QUALIFICACAO_RESPONSAVEL = f.id
    LEFT join infra-itaborai.mds_cnpj.d_porte as g on b.COD_PORTE_EMPRESA = g.id_porte
    LEFT JOIN infra-itaborai.mds_cnpj.d_pais as h on a.pais = h.id
    LEFT JOIN infra-itaborai.mds_cnpj.d_natjuridica as i on b.COD_NATUREZA_JUR = i.id
    left join infra-itaborai.mds_cnpj.d_municipio as j on a.MUNICIPIO = j.id
    left join infra-itaborai.mds_cnpj.d_motivo as k on a.motivo_situacao = k.id_motivo
    left JOIN infra-itaborai.mds_cnpj.d_cnae as l on a.CNAE_PRINCIPAL=l.id 

WHERE a.MUNICIPIO = '0309' 
AND a.situacao_cadastral IN ('02', '04')
AND CNAE LIKE '47%'
--AND d.descricao LIKE '%COMÉRCIO VAREJISTA%'

AND E_MAIL IS NOT NULL
-- AND DATA_INICIO_ATIVIDADE >= '2024-01-01' 
-- ORDER BY A.DATA_INICIO_ATIVIDADE DESC

"""
print("Extraindo dados...")
rfb = pandas_gbq.read_gbq(queryrfb, project_id=project_id)
print(f"Total de linhas: {len(rfb)}")

# https://www.fazenda.mg.gov.br/governo/assuntos_municipais/codigomunicipio/codmunicoutest_rr.html

Extraindo dados...
Downloading: 100%|██████████|
Total de linhas: 339


In [97]:
rfb

,CNPJ_COMPLETO,razao_social,telefone,email,CNAE,DESCRICAO_CNAE
0,17940233000177,E DO N MOURA COMERCIO LTDA,9535421397,edelria30@hatmail.com,4711302,"COMÉRCIO VAREJISTA DE MERCADORIAS EM GERAL, CO..."
1,26288463000194,A R COMERCIO DE COMBUSTIVEL LTDA,9581144444,rodrigo_cal@hotmail.com.br,4731800,COMÉRCIO VAREJISTA DE COMBUSTÍVEIS PARA VEÍCUL...
2,13909145000151,RONICLEITON DA SILVA MELO 00063321297,9591311706,ronnymello1986@gmail.com,4755502,COMÉRCIO VAREJISTA DE ARTIGOS DE ARMARINHO
3,26349289000142,LUCIMARA GOMES DA CRUZ 02755650257,9599705683,adevaldomarques25@gmail.com,4721103,COMÉRCIO VAREJISTA DE LATICÍNIOS E FRIOS
4,26557559000100,ROBSON SANTOS DA ROCHA 00254463207,9591687968,robsonsantosdarocha4@gmail.com,4753900,COMÉRCIO VAREJISTA ESPECIALIZADO DE ELETRODOMÉ...
...,...,...,...,...,...,...
334,33861342000155,NaN,9591229775,jordelrufino4@gmail.com,4752100,COMÉRCIO VAREJISTA ESPECIALIZADO DE EQUIPAMENT...
335,65010279000130,KELRYLANNE OLIVEIRA DOS SANTOS,9598415638,kelrylanneo@gmail.com,4781400,COMÉRCIO VAREJISTA DE ARTIGOS DO VESTUÁRIO E A...
336,31795646000136,NaN,9581047762,ismarlyson98@hotmail.com,4781400,COMÉRCIO VAREJISTA DE ARTIGOS DO VESTUÁRIO E A...
337,41734579000101,DIURVIS ZAGUEY BANDRES CARAMO 70774691239,9584003485,diurvisbandres27@gmail.com,4755502,COMÉRCIO VAREJISTA DE ARTIGOS DE ARMARINHO


In [98]:
rfb.to_csv(r'C:\Users\Nelio\Documents\cnpjmucajai.csv', index=False)